In [0]:

from pyspark.sql import functions as F

df = spark.table(
    "hackathon.shared_datasets.fraud_busters_features"
)

print(df.count())
print(df.columns)


590540
['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V41', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'V61', 'V62', 'V63', 'V64', 'V65', 'V66', 'V67', 'V68', 'V69', 'V70', 'V71', 'V72', 'V73', 'V74', 'V75', 'V76', 'V77', 'V78', 'V79', 'V80', 'V

In [0]:
numeric_features = [
    "TransactionAmt",
    "log_transaction_amt",
    "TransactionDT",
    "hour_of_day",
    "day_number",
    "card1",
    "card2",
    "card3",
    "card5",
    "addr1",
    "addr2",
    "dist1",
    "dist2"
]

categorical_features = [
    "ProductCD",
    "card4",
    "card6",
    "P_emaildomain",
    "DeviceType"
]





In [0]:
target_column = "isFraud"

In [0]:

selected_columns = (
    numerical_features
    + categorical_features
    + [target_column]
)

pdf = (
    df
    .select(selected_columns)
    .toPandas()
)

print(pdf.shape)


(590540, 19)


In [0]:

X = pdf.drop(
    columns=[target_column]
)

y = pdf[target_column].astype(int)


In [0]:
print(X.dtypes)

TransactionAmt         float64
log_transaction_amt    float64
TransactionDT            int64
hour_of_day              int32
day_number               int32
card1                    int64
card2                  float64
card3                  float64
card5                  float64
addr1                  float64
addr2                  float64
dist1                  float64
dist2                  float64
ProductCD               object
card4                   object
card6                   object
P_emaildomain           object
DeviceType              object
dtype: object


In [0]:
object_cols = X.select_dtypes(include=["object"]).columns

for col in object_cols:
    X[col] = (
        X[col]
        .fillna("missing")
        .astype("category")
        .cat.codes
    )


In [0]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print(f"Train rows: {len(X_train):,}")
print(f"Test rows:  {len(X_test):,}")


Train rows: 413,378
Test rows:  177,162


In [0]:
print(X.dtypes[X.dtypes == "object"])

Series([], dtype: object)


In [0]:

from sklearn.ensemble import HistGradientBoostingClassifier
model = HistGradientBoostingClassifier(

    max_depth=8,

    learning_rate=0.05,

    max_iter=300,

    random_state=42
)

model.fit(
    X_train,
    y_train
)



HistGradientBoostingClassifier(learning_rate=0.05, max_depth=8, max_iter=300,
                               random_state=42)

In [0]:

y_pred = model.predict(X_test)

y_prob = model.predict_proba(
    X_test
)[:,1]


In [0]:

from sklearn.metrics import (

    roc_auc_score,

    average_precision_score,

    precision_score,

    recall_score,

    f1_score,

    classification_report
)


In [0]:
roc_auc = roc_auc_score(
    y_test,
    y_prob
)

pr_auc = average_precision_score(
    y_test,
    y_prob
)

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)

In [0]:
print("="*50)

print("STRUCTURED FRAUD BASELINE")

print("="*50)

print(f"ROC-AUC       : {roc_auc:.4f}")

print(f"PR-AUC        : {pr_auc:.4f}")

print(f"Precision     : {precision:.4f}")

print(f"Recall        : {recall:.4f}")

print(f"Fraud F1      : {f1:.4f}")

STRUCTURED FRAUD BASELINE
ROC-AUC       : 0.8768
PR-AUC        : 0.4141
Precision     : 0.8359
Recall        : 0.1118
Fraud F1      : 0.1972
